In [1]:
%pip install torch torchvision scipy numpy


Note: you may need to restart the kernel to use updated packages.


In [4]:
import h5py
import numpy as np
import torch
from torch.utils.data import Dataset
import torchvision.transforms as transforms
import os

class TumorDataset(Dataset):
    def __init__(self, mat_dir, transform=None):
        self.mat_dir = mat_dir
        self.files = sorted([f for f in os.listdir(mat_dir) if f.endswith(".mat")])
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        filepath = os.path.join(self.mat_dir, self.files[idx])

        with h5py.File(filepath, "r") as f:
            cj = f["cjdata"]

            img = np.array(cj["image"]).astype(np.float32)

            # FIX: extract scalar label from nested array
            label = np.array(cj["label"]).astype(int).item()

            # map label 4 → 0
            if label == 4:
                label = 0

        # normalize image
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)

        # convert grayscale → 3-channel
        img = np.stack([img, img, img], axis=-1)

        if self.transform:
            img = self.transform(img)

        return img, label


In [5]:
from torch.utils.data import DataLoader, random_split
import torch.nn as nn
import torchvision.models as models

mat_dir = "/Users/ha/Documents/Final Year project/Dataset/brainTumorDataPublic/gaussian_sigma1_threshold527"

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((224, 224)),
    transforms.Normalize([0.5], [0.5])
])

dataset = TumorDataset(mat_dir, transform)

# 80/20 train-test split
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_ds, test_ds = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)

device = "mps" if torch.backends.mps.is_available() else "cpu"


model = models.densenet121(weights=None)  # set weights="IMAGENET1K_V1" for transfer learning
model.classifier = nn.Linear(model.classifier.in_features, 4)  # class 0–3

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


In [6]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
import numpy as np

def calculate_specificity(y_true, y_pred, num_classes):
    """
    Calculates weighted specificity for multiclass classification.
    """
    cm = confusion_matrix(y_true, y_pred, labels=range(num_classes))
    specificities = []
    support = []
    
    for i in range(num_classes):
        # For class i:
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        tn = cm.sum() - (tp + fp + fn)
        
        # Specificity = TN / (TN + FP)
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        specificities.append(spec)
        support.append(cm[i, :].sum())
        
    # Return weighted average based on class support (similar to sklearn's 'weighted' logic)
    return np.average(specificities, weights=support)

def train(model, loader, optimizer, criterion, device, num_classes=3):
    model.train()
    total_loss = 0
    all_labels = []
    all_preds = []

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(predicted.cpu().numpy())

    # Calculate metrics
    acc = 100 * np.mean(np.array(all_preds) == np.array(all_labels))
    prec = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    rec = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    spec = calculate_specificity(all_labels, all_preds, num_classes)

    print(f"TRAIN | Loss: {total_loss/len(loader):.4f} | Acc: {acc:.2f}% | Prec: {prec:.3f} | Rec: {rec:.3f} | F1: {f1:.3f} | Spec: {spec:.3f}")

def evaluate(model, loader, device, num_classes=3):
    model.eval()
    all_labels = []
    all_preds = []
    
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            _, predicted = torch.max(outputs, 1)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())

    # Calculate metrics
    acc = 100 * np.mean(np.array(all_preds) == np.array(all_labels))
    prec = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    rec = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    spec = calculate_specificity(all_labels, all_preds, num_classes)

    print(f"TEST  | Acc: {acc:.2f}% | Prec: {prec:.3f} | Rec: {rec:.3f} | F1: {f1:.3f} | Spec: {spec:.3f}")
    return f1

# --- Main Training Loop ---
EPOCHS = 100
best_f1 = 0.0

for epoch in range(EPOCHS):
    print(f"\n--- Epoch {epoch+1}/{EPOCHS} ---")
    train(model, train_loader, optimizer, criterion, device, num_classes=3)
    current_f1 = evaluate(model, test_loader, device, num_classes=3)
    
    if current_f1 > best_f1:
        best_f1 = current_f1
        torch.save(model.state_dict(), "best_model_f1.pth")
        print(f">> New best F1-Score: {best_f1:.4f}. Model weights saved.")


--- Epoch 1/100 ---
TRAIN | Loss: 0.4531 | Acc: 81.96% | Prec: 0.817 | Rec: 0.820 | F1: 0.818 | Spec: 0.810
TEST  | Acc: 89.79% | Prec: 0.903 | Rec: 0.898 | F1: 0.896 | Spec: 0.909
>> New best F1-Score: 0.8961. Model weights saved.

--- Epoch 2/100 ---
TRAIN | Loss: 0.2435 | Acc: 91.01% | Prec: 0.910 | Rec: 0.910 | F1: 0.910 | Spec: 0.917
TEST  | Acc: 90.69% | Prec: 0.913 | Rec: 0.907 | F1: 0.907 | Spec: 0.937
>> New best F1-Score: 0.9073. Model weights saved.

--- Epoch 3/100 ---
TRAIN | Loss: 0.1690 | Acc: 93.96% | Prec: 0.940 | Rec: 0.940 | F1: 0.940 | Spec: 0.947
TEST  | Acc: 95.35% | Prec: 0.954 | Rec: 0.953 | F1: 0.953 | Spec: 0.961
>> New best F1-Score: 0.9534. Model weights saved.

--- Epoch 4/100 ---
TRAIN | Loss: 0.1319 | Acc: 95.42% | Prec: 0.954 | Rec: 0.954 | F1: 0.954 | Spec: 0.960
TEST  | Acc: 95.61% | Prec: 0.956 | Rec: 0.956 | F1: 0.956 | Spec: 0.940
>> New best F1-Score: 0.9558. Model weights saved.

--- Epoch 5/100 ---
TRAIN | Loss: 0.1009 | Acc: 96.43% | Prec: 0.96

In [14]:
ECPOS = 20
print("4.gaussian_sigma1_threshold527 training Start")

for i in range(ECPOS):
    print(f"Epoch {i+1}/{ECPOS}")
    train(model, train_loader)
    evaluate(model, test_loader)


4.gaussian_sigma1_threshold527 training Start
Epoch 1/20


KeyboardInterrupt: 

In [ ]:
# Save the entire model (full model object)
torch.save(model, "4.gaussian_sigma1_threshold527.pth")